# Day 10 - Model Save/Load + ModelCheckpoint


## 1. 데이터 및 모델 준비


In [ ]:
import tensorflow as tf
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import numpy as np
import os

tf.random.set_seed(42)
data = load_diabetes()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1)
])
model.compile(optimizer='adam', loss='mse')
print("Model ready")


## 2. ModelCheckpoint + EarlyStopping 학습


In [ ]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_diabetes.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)
early = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=32,
    validation_split=0.2,
    callbacks=[checkpoint, early],
    verbose=0
)
print("Training completed")


## 3. 모델 저장 / 불러오기 테스트


In [ ]:
# 전체 모델 저장
model.save('diabetes_full.keras')
print("Full model saved: diabetes_full.keras")

# weights만 저장
model.save_weights('diabetes_weights.weights.h5')
print("Weights saved: diabetes_weights.weights.h5")

# 최고 성능 모델 불러오기
loaded = tf.keras.models.load_model('best_diabetes.keras')
y_pred = loaded.predict(X_test, verbose=0).flatten()
print("Loaded model R2:", round(r2_score(y_test, y_pred), 4))
print("Save / Load successful!")

# 파일 존재 확인
for f in ['best_diabetes.keras', 'diabetes_full.keras', 'diabetes_weights.weights.h5']:
    print(f"  {f}: exists={os.path.exists(f)}")
